<a href="https://colab.research.google.com/github/raulmabcn/DataAnalyst/blob/main/Foundations/Python/Exercices/Bloque4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import pandas as pd
data_h = pd.read_csv('hospitals.csv', sep=';', encoding='latin-1')
data_h.info()
data_p = pd.read_csv('pacientes.csv', sep=';', encoding='latin-1')
data_p.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   hospital_id                 170 non-null    int64  
 1   nombre                      170 non-null    object 
 2   localidad                   170 non-null    object 
 3   provincia                   170 non-null    object 
 4   comunidad_autonoma          170 non-null    object 
 5   presupuesto_anual_millones  170 non-null    float64
 6   num_camas                   170 non-null    int64  
 7   num_medicos                 170 non-null    int64  
 8   indice_satisfaccion         170 non-null    float64
 9   porcentaje_ocupacion        170 non-null    float64
dtypes: float64(3), int64(3), object(4)
memory usage: 13.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5060 entries, 0 to 5059
Data columns (total 8 columns):
 #   Column               Non-Null Count  

Detecta comunidades autónomas donde:
  El crecimiento de pacientes (respecto a la media nacional) es alto pero el crecimiento de camas no lo es. (Define tú la métrica de crecimiento).



crecimiento_pacientes = (total_pacientes_comunidad - media_pacientes_hospital * num_hospitales )/ (media_pacientes_hospital * num_hospitales)

crecimiento_camas = (total_camas_comunidad - media_camas_hospital * num_hospitales )/ (media_camas_hospital * num_hospitales)

In [26]:
pacientes = data_p.groupby('hospital_id', as_index=False).agg(
    num_pacientes = ('paciente_id', 'count')
)

data_h_p = data_h.merge( pacientes, on = 'hospital_id', how= 'left' )
data_h_p['num_pacientes'] = data_h_p['num_pacientes'].fillna(0)

m_pacientes = data_h_p['num_pacientes'].mean()
m_camas     = data_h_p['num_camas'].mean()

data_co = data_h_p.groupby('comunidad_autonoma').agg(
  t_pacientes   = ('num_pacientes', 'sum'),
  t_camas       = ('num_camas', 'sum'),
  n_hospitales  = ('hospital_id', 'count')
)

data_co['crc_pacientes']  = (data_co['t_pacientes'] - m_pacientes * data_co['n_hospitales'] ) / ( m_pacientes * data_co['n_hospitales'] )
data_co['crc_camas']      = ( data_co['t_camas'] - m_camas * data_co['n_hospitales'] ) / ( m_camas * data_co['n_hospitales'] )

data_co.loc[ data_co['crc_pacientes'] > data_co['crc_camas'] ].index

Index(['Andalucia', 'Aragon', 'Asturias', 'Castilla-La Mancha', 'Ceuta',
       'Comunidad de Madrid', 'Extremadura', 'La Rioja', 'Melilla'],
      dtype='object', name='comunidad_autonoma')

Identifica hospitales atípicos usando:
* Z-score sobre indice_satisfaccion
* IQR



In [27]:
media = data_h['indice_satisfaccion'].mean()
std   = data_h['indice_satisfaccion'].std()
data_h['z_score'] = ( data_h['indice_satisfaccion']- media ) / std

outliers_z = data_h[data_h['z_score'].abs() > 1.8 ]
outliers_z

q1 = data_h['indice_satisfaccion'].quantile(0.25)
q3 = data_h['indice_satisfaccion'].quantile(0.75)

iqr = q3 - q1
modifier = 0.5 #alterardo seria 1.5

q_lower = q1 - modifier * iqr
q_upper = q3 + modifier * iqr


outliers_q = data_h[ ( data_h['indice_satisfaccion'] < q_lower ) | ( data_h['indice_satisfaccion'] > q_upper )]
outliers_q

,hospital_id,nombre,localidad,provincia,comunidad_autonoma,presupuesto_anual_millones,num_camas,num_medicos,indice_satisfaccion,porcentaje_ocupacion,z_score
17,18,Hospital Universitario de Jerez,Jerez de la Frontera,Cadiz,Andalucia,232.89,805,456,9.8,68.7,1.649929
22,23,Hospital de Alta Resolucion de la Costa Occide...,Lepe,Huelva,Andalucia,261.85,304,462,9.7,50.4,1.609871
25,26,Hospital Costa del Sol,Marbella,Malaga,Andalucia,248.10,610,178,9.7,51.4,1.609871
45,46,Hospital Universitario San Agustin,Aviles,Asturias,Asturias,282.13,465,129,9.7,70.7,1.609871
51,52,Hospital de Manacor,Manacor,Mallorca,Islas Baleares,215.55,789,127,1.3,50.8,-1.755023
76,77,Hospital Virgen de la Salud,Toledo,Toledo,Castilla-La Mancha,45.72,297,80,1.2,79.0,-1.795081
91,92,Hospital Universitario Vall d'Hebron,Barcelona,Barcelona,Cataluña,496.44,279,109,1.1,89.9,-1.835140
101,102,Hospital Comarcal del Pallars,Tremp,Lleida,Cataluña,171.75,626,280,1.1,79.8,-1.835140
105,106,Hospital Clinico Universitario de Valencia,Valencia,Valencia,Comunidad Valenciana,61.59,457,139,9.9,65.6,1.689987
117,118,Hospital Universitario de Badajoz,Badajoz,Badajoz,Extremadura,185.08,175,145,1.3,97.4,-1.755023


Construye una tabla resumen final por comunidad autónoma con:
* total hospitales
* total camas
* total pacientes
* ocupación media
* satisfacción media
* ratio pacientes/cama
* ratio pacientes/médico

Ordena por eficiencia (define tú eficiencia).

In [36]:
pacientes = data_p.groupby('hospital_id', as_index=False).agg(
    num_pacientes = ('paciente_id', 'count')
)

data_h_p = data_h.merge( pacientes, on = 'hospital_id', how= 'left' )
data_h_p['num_pacientes'] = data_h_p['num_pacientes'].fillna(0)

data_co = data_h_p.groupby('comunidad_autonoma').agg(
  n_hospitales    = ('hospital_id', 'count'),
  t_camas         = ('num_camas', 'sum'),
  t_medicos       = ('num_medicos', 'sum'),
  t_pacientes     = ('num_pacientes', 'sum'),
  m_ocupacion     = ('porcentaje_ocupacion', 'mean'),
  m_satisfaccion  = ('indice_satisfaccion', 'mean')
)

data_co['r_p_c'] = data_co['t_pacientes'] / data_co['t_camas']
data_co['r_p_m'] = data_co['t_pacientes'] / data_co['t_medicos']

data_co.sort_values(['m_satisfaccion'], ascending=False)

,n_hospitales,t_camas,t_medicos,t_pacientes,m_ocupacion,m_satisfaccion,r_p_c,r_p_m
comunidad_autonoma,,,,,,,,
Navarra,4,2617,988,108,74.800000,7.775000,0.041269,0.109312
Asturias,6,1980,1556,176,82.400000,7.383333,0.088889,0.113111
Islas Canarias,9,4858,2502,252,73.055556,7.055556,0.051873,0.100719
Aragon,10,4661,3263,307,81.810000,6.860000,0.065866,0.094085
Melilla,1,652,127,47,88.400000,6.500000,0.072086,0.370079
Ceuta,1,208,276,34,88.500000,6.300000,0.163462,0.123188
Comunidad Valenciana,12,6547,3297,363,77.575000,6.258333,0.055445,0.110100
Pais Vasco,11,6689,3446,336,75.436364,6.190909,0.050232,0.097504
Castilla y Leon,11,5948,2676,280,75.581818,6.054545,0.047075,0.104634
